```bash
pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[OptimalTransport]'
```

- Introduction (/3)
- Preparation et analyse des données (/3)
- Application des méthodes de pre processing (/5)
- Application des méthodes de post processing (/5)
- Analyse, compréhension (/3)
- Conclusion (/1)

## Introduction

nous disposons désormais des images elles-mêmes, ce qui nous permet d'entraîner un véritable modèle de prédiction. L’objectif est de construire un pipeline complet comportant :
- un prétraitement visant à atténuer les biais avant l'entraînement (par exemple via l'algorithme LFR),
- un modèle de classification basé sur les images pour prédire les maladies,
- un post-traitement appliqué aux prédictions pour corriger d’éventuelles inégalités restantes (via l'algorithme Equalized Odds Postprocessing par exemple).

Ce rapport présente la mise en œuvre de ce pipeline, les défis rencontrés, et une évaluation de l'impact des différentes étapes sur la qualité et l’équité des prédictions.

In [6]:
import utils
import os
import pandas as pd
from constants import *
from train_classifieur import train_classifier, pred_classifier
from aif360.datasets import BinaryLabelDataset
import plotly.express as px


utils.load_env_file()
data_dir = os.getenv("DATA_DIR", "data/default/")
og_metadata_filename="original_metadata.csv"
og_metadata_path = data_dir + og_metadata_filename
pred_output_dir="./expe_log/selected_data/"
print("Travaille sur : ", data_dir)
print("Output en : ", pred_output_dir)
print(og_metadata_path)

Travaille sur :  ./data/SAILLANT_ARTHUR/selected_data/
Output en :  ./expe_log/selected_data/
./data/SAILLANT_ARTHUR/selected_data/original_metadata.csv


In [7]:
# variables et fonctions importante 

map_genre = {"M": 0, "F": 1}
map_viewposition = {"AP": 0, "PA": 1}
map_pred = {"sain": 0, "malade": 1}

fav_lbl = map_pred["sain"]
unfav_lbl = map_pred["malade"]
protected_attributes = ['Patient Gender', '+40ans']

protected_attribute = protected_attributes[1]

priviliged_group = 0
unpriviliged_group = 1


unprivileged_groups = [{protected_attribute: unpriviliged_group}]
privileged_groups = [{protected_attribute: priviliged_group}]


In [8]:

def convert_to_all_numerical(df):
    # Define paths to the train repository
    train_sain_path = data_dir+"/train/sain"
    train_malade_path = data_dir+"/train/malade"

    # Get the list of image filenames in the train repository
    train_images = set(os.listdir(train_sain_path) + os.listdir(train_malade_path))

    df.columns = df.columns.str.strip()
    if "in_train" not in df.columns:
        df["in_train"] = df["Image Index"].apply(lambda x: 1 if x in train_images else 0)
    if 'Finding Labels' in df.columns:
        df_ohe = df['Finding Labels'].str.get_dummies(sep='|').astype(bool)
        df = df.drop(columns=['Finding Labels']).join(df_ohe)
    if "preds" in df.columns and not pd.api.types.is_numeric_dtype(df["preds"]):
        df["preds"] = df["preds"].map({"sain": 0, "malade": 1})
    if "labels" in df.columns and not pd.api.types.is_numeric_dtype(df["labels"]):
        df["labels"] = df["labels"].map({"sain": 0, "malade": 1})
    if not pd.api.types.is_numeric_dtype(df["Patient Gender"]):
        df["Patient Gender"] = df["Patient Gender"].map(map_genre)
    if "View Position" in df.columns and not pd.api.types.is_numeric_dtype(df["View Position"]):
        df["View Position"] = df["View Position"].map(map_viewposition)
    if "+40ans" not in df.columns:
        df["+40ans"] = (df["Patient Age"] > 40).astype(int) 
    return df

In [9]:

from aif360.sklearn.metrics import *


def get_group_metrics(
    y_true,
    y_pred=None,
    prot_attr=None,
    priv_group=1,
    pos_label=1,
    sample_weight=None,
):
    group_metrics = {}
    group_metrics["base rate"] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["SPD"] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["DI"] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    if not y_pred is None:
        group_metrics["equal_opportunity_difference"] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["average_odds_difference"] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["conditional_demographic_disparity"] = conditional_demographic_disparity(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["smoothed_edf"] = smoothed_edf(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["df_bias_amplification"] = df_bias_amplification(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
    return group_metrics


In [10]:
def train_and_predict(metadata_csv, outputcsv, force_training=False):
    os.makedirs(pred_output_dir, exist_ok=True)
    csv_out = os.path.join(pred_output_dir, outputcsv)
    csv_in=data_dir+metadata_csv
    if force_training or not os.path.exists(csv_out):
        print("Entrainement du classifieur...")
        ckpt_path, ckpt_score = train_classifier(
            logdir=pred_output_dir,
            datadir=data_dir,
            csv=csv_in,
        )
        print("Génerations des predictions...")
        pred_classifier(
            datadir=data_dir,
            csv_in=csv_in,
            csv_out=csv_out,
            ckpt_path=ckpt_path
        )
    else:
        print(f"Les prédiction existent déjà à {csv_out} -- abandon de l'entraînement")

def intoBinaryLabelDataset(df):
    manquantes = [attribute for attribute in protected_attributes if attribute not in df.columns]

    if manquantes:
        raise ValueError(f"Les colonnes protégées suivantes sont manquantes dans le dataset : {', '.join(manquantes)}")

    dataset = BinaryLabelDataset(
        favorable_label=fav_lbl,  # "Sain" est la classe favorable
        unfavorable_label=unfav_lbl,  # "Malade" est la classe défavorable
        df=df,
        label_names=["labels"],
        protected_attribute_names=protected_attributes
    )
    return dataset

def getMetric(df, prot_attr):
    if isinstance(prot_attr, list) :
        raise RuntimeError("On ne peut pas faire de metriquesurplusieur attr protegé")
    df = convert_to_all_numerical(df)
    preds = df["preds"]
    labels= df["labels"]
    weights = df["WEIGHTS"]

    metrics_after_reweight = get_group_metrics(
        y_true=labels,
        y_pred=preds,
        prot_attr=df[prot_attr],
        priv_group=1,
        pos_label=1,
        sample_weight=weights
    )
    return metrics_after_reweight

    


## Préparations des données

Notamment pour les converitir dans un ``BinaryLabelDataset``

In [11]:
df = pd.read_csv(og_metadata_path)

print(df.columns)
imageid_df = df.copy()[["Image Index", patientid]]
original_df = df.copy()
df = convert_to_all_numerical(df)

df.head() # Y'a toujours Image index !!

Index(['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID',
       'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width',
       'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'WEIGHTS'],
      dtype='object')


,Image Index,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],...,Fibrosis,Hernia,Infiltration,Mass,No Finding,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,+40ans
0,00000042_006.png,6,42,71,0,0,3056,2544,0.139000,0.139000,...,False,False,True,False,False,False,False,False,False,1
1,00000048_000.png,0,48,46,1,1,2834,2641,0.143000,0.143000,...,False,False,False,False,True,False,False,False,False,1
2,00000096_000.png,0,96,67,1,1,2646,2829,0.143000,0.143000,...,False,False,False,False,False,False,False,False,False,1
3,00000097_000.png,0,97,83,1,1,2021,1865,0.194311,0.194311,...,False,False,False,False,True,False,False,False,False,1
4,00000100_000.png,0,100,60,0,1,2500,2048,0.171000,0.171000,...,False,False,True,False,False,False,False,False,False,1


In [12]:
# recupperer les predictions sans aucun changement
train_and_predict(og_metadata_filename, "original_preds.csv", force_training=True)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Entrainement du classifieur...


/home/arthur/Code/python/fairness/projet/contrarielapulpedemafairness/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/arthur/Code/python/fairness/projet/contrarielapulpedemafairness/expe_log/selected_data exists and is not empty.
/home/arthur/Code/python/fairness/projet/contrarielapulpedemafairness/.venv/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name  | Type                  | Params | Mode 
--------------------------------------------------------
0 | model | ResNet                | 11.2 M | train
1 | bcm   | BinaryConfusionMatrix | 0      | train
--------------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.710    Total estimated model params size (MB)
69        Modules in train mode
0         Modul

Start training


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

End of training 840.0590078830719
Génerations des predictions...
Start prediction on train dataset
Predictions done in 68.79125308990479
Start prediction on validation dataset
Predictions done in 91.96876430511475
0.7052116388851083
0.7086666666666667


In [13]:
preddf = pd.read_csv(pred_output_dir+"original_preds.csv")
preddf = convert_to_all_numerical(preddf)

metrics_before_training = getMetric(preddf, protected_attribute)

def compare_to_base_preds(metrics_after):
    for metric in metrics_before_training.keys():
        before = metrics_before_training[metric]
        after = metrics_after[metric]
        change = after - before
        print(f"{before:.4f} ---- {metric} ---> {after:.4f}, (diff = {change:.4f})")


## Analyse

On va surtout s'interesser à l'âge 

In [14]:
utils.plot_age_dist(original_df)

In [15]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_confusion_matrices_side_by_side(df, group_column, labels=["sain", "malade"], normalize=False):
    y_true = df["labels"].values
    y_pred = df["preds"].values
    unique_groups = df[group_column].unique()
    n_groups = len(unique_groups)
    
    fig = make_subplots(
        rows=1,
        cols=n_groups,
        subplot_titles=[f"{group_column} = {val}" for val in unique_groups],
        horizontal_spacing=0.25 
    )

    for i, group_value in enumerate(unique_groups):
        group_df = df[df[group_column] == group_value]
        y_true_group = y_true[group_df.index]
        y_pred_group = y_pred[group_df.index]

        cm = confusion_matrix(y_true_group, y_pred_group)

        if normalize:
            cm = cm.astype('float') / len(y_true_group) * 100
        z_text = [[f"{val:.2f}%" if normalize else str(int(val)) for val in row] for row in cm]
        fig.add_trace(
            go.Heatmap(
                z=cm, x=labels, y=labels,
                colorscale="Blues",
                showscale=False,
                zmin=0, zmax=100 if normalize else None,
                text=z_text, texttemplate="%{text}", hoverinfo="z"
            ),
            row=1, col=i+1
        )

        fig.update_xaxes(title_text="Prédiction", row=1, col=i+1)
        fig.update_yaxes(title_text="Vérité", row=1, col=i+1, autorange="reversed")

    fig.update_layout(
        title_text="Matrices de Confusion par Groupe",
        height=400,
        width=420 * n_groups  # plus de largeur pour espacer
    )
    
    fig.show()


In [16]:
preddf['+40ans'] = preddf['Patient Age'] >= 40
plot_confusion_matrices_side_by_side(
    df=preddf,
    group_column='+40ans',
    labels=["sain", "malade"],
)

In [17]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix

error_rate_df = pd.DataFrame(columns=['method', 'global', '+40ans', '-40ans', 'M', 'F'])

def add_error_rate(df, method):
    global error_rate_df  

    y_true = df["labels"].values
    y_pred = df["preds"].values
    cm = confusion_matrix(y_true, y_pred)
    total = cm.sum()
    correct = np.trace(cm)
    global_error_rate = (total - correct) / total * 100
    
    df['age_binary'] = (df['Patient Age'] > 40).astype(int)
    def compute_error_rate(group_df):
        cm = confusion_matrix(group_df["labels"].values, group_df["preds"].values)
        return (cm.sum() - np.trace(cm)) / cm.sum() * 100
    age_error_rate = df.groupby('age_binary').apply(compute_error_rate)
    sex_error_rate = df.groupby('Patient Gender').apply(compute_error_rate)
    new_row = pd.DataFrame({
        'method': [method],
        'global': [global_error_rate],
        '+40ans': [age_error_rate.get(1, np.nan)],  # Age > 40
        '-40ans': [age_error_rate.get(0, np.nan)],  # Age <= 40
        'M': [sex_error_rate.get(1, np.nan)],       # Sexe = M (homme)
        'F': [sex_error_rate.get(0, np.nan)]        # Sexe = F (femme)
    })

    error_rate_df = pd.concat([error_rate_df, new_row], ignore_index=True)
    

In [18]:
add_error_rate(preddf, 'Normal')
error_rate_df

/tmp/ipykernel_95114/3144252713.py:21: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_95114/3144252713.py:22: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_95114/3144252713.py:32: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result 

,method,global,+40ans,-40ans,M,F
0,Normal,29.133333,31.083845,25.478927,25.96291,31.914894


## pre processing

#### Pre pre processing

In [19]:

og_preddf = pd.read_csv(pred_output_dir+"original_preds.csv")
og_preddf = convert_to_all_numerical(og_preddf)




filtered_df = og_preddf.drop(["View Position", "Finding Labels", "Image Index"], axis=1, errors="ignore")
train_df = filtered_df[filtered_df["in_train"]==1].copy().reset_index()
test_df = filtered_df[filtered_df["in_train"]==0].copy().reset_index()

dataset = intoBinaryLabelDataset(filtered_df)
train_dataset = intoBinaryLabelDataset(train_df)
test_dataset = intoBinaryLabelDataset(test_df)


a=len(train_dataset.instance_weights)
b=len(test_dataset.instance_weights)
c=len(dataset.instance_weights)
assert(a+b==c)
# train_df

print(f"taille du train : {len(train_df)}")
print(f"taille du test : {len(test_df)}")

taille du train : 1125
taille du test : 375


#### reweight

In [20]:
sensitive_attr = "+40ans"
unprivileged_groups, privileged_groups=[{sensitive_attr: 0}], [{sensitive_attr: 1}]

In [21]:
from aif360.algorithms.preprocessing import Reweighing

rw = Reweighing(unprivileged_groups, privileged_groups)
rw.fit(train_dataset)
transformed_dataset = rw.transform(dataset)

csv_df = original_df.copy()
csv_df["WEIGHTS"] = transformed_dataset.instance_weights
csv_df.to_csv(data_dir+"/"+"reweighted_metadata.csv", index=False)


In [22]:
train_and_predict("reweighted_metadata.csv", "reweighted_preds.csv")

Les prédiction existent déjà à ./expe_log/selected_data/reweighted_preds.csv -- abandon de l'entraînement


In [23]:
rw_pred = pd.read_csv(pred_output_dir+"reweighted_preds.csv")
rw_pred = convert_to_all_numerical(rw_pred)
metrics_after_reweight = getMetric(rw_pred, sensitive_attr)

In [24]:
compare_to_base_preds(metrics_after_reweight)

0.4573 ---- base rate ---> 0.4573, (diff = -0.0000)
-0.2284 ---- SPD ---> 0.0068, (diff = 0.2352)
0.5621 ---- DI ---> 1.0186, (diff = 0.4566)
-0.1536 ---- equal_opportunity_difference ---> 0.0206, (diff = 0.1743)
-0.1680 ---- average_odds_difference ---> 0.0113, (diff = 0.1793)
-0.0639 ---- conditional_demographic_disparity ---> 0.0020, (diff = 0.0659)
0.5747 ---- smoothed_edf ---> 0.0187, (diff = -0.5560)
0.2292 ---- df_bias_amplification ---> 0.0006, (diff = -0.2286)


In [25]:
add_error_rate(rw_pred, "Reweight")
error_rate_df

/tmp/ipykernel_95114/3144252713.py:21: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_95114/3144252713.py:22: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,method,global,+40ans,-40ans,M,F
0,Normal,29.133333,31.083845,25.478927,25.96291,31.914894
1,Rewight,27.400000,28.834356,24.712644,26.81883,27.909887


In [26]:
plot_confusion_matrices_side_by_side(
    df=rw_pred,
    group_column='+40ans',
    labels=["sain", "malade"],
)
# plot_confusion_matrix_by_group(rw_pred["labels"], rw_pred["preds"], rw_pred, group_columns=["+40ans"], labels=[0, 1])

#### DIR

In [27]:
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import DisparateImpactRemover
import pandas as pd

def apply_disparate_impact_remover(original_df, repair_level=1.0):
    label_col = "labels"

    protected_attr = "+40ans"

    # Colonnes à garder pour la réparation
    dir_features = ["Patient Age", "Patient Gender", protected_attr, label_col, "WEIGHTS"] 
    
    df_dir = original_df[dir_features].copy()
    dataset = BinaryLabelDataset(
        df=df_dir,
        label_names=[label_col],
        protected_attribute_names=[protected_attr]
    )

    # pour les restaurer ensuite
    patient_ids = original_df["Patient ID"].astype(str).tolist()
    dataset.instance_names = [[pid] for pid in patient_ids]

    dir = DisparateImpactRemover(sensitive_attribute=protected_attr, repair_level=repair_level)
    repaired_dataset = dir.fit_transform(dataset)
    repaired_df = pd.DataFrame(
        data=repaired_dataset.features,
        columns=repaired_dataset.feature_names
    )
    repaired_df[label_col] = repaired_dataset.labels
    # on remet les ids et les images
    repaired_df["Patient ID"] = [int(pid[0]) for pid in repaired_dataset.instance_names]
    imageid_df["Patient ID"] = imageid_df["Patient ID"].astype(int)
    repaired_df = repaired_df.merge(imageid_df, on="Patient ID", how="left")

    columns_to_add = ["in_train"]
    for col in columns_to_add:
        repaired_df[col] = original_df[col].values

    repaired_df["+40ans"] = (repaired_df["Patient Age"] >= 40).astype(int)
    return repaired_df


In [28]:
# pournepas utiliser d'info du datatest dans le train -> data leakage

repaired_train_df = apply_disparate_impact_remover(train_df)
repaired_test_df =  apply_disparate_impact_remover(test_df)

repaired_df = pd.concat([repaired_train_df, repaired_test_df], ignore_index=True)

repaired_df.to_csv(data_dir+"/"+"dir_metadata.csv", index=False)


In [29]:
train_and_predict("dir_metadata.csv", "dir_preds.csv")

Les prédiction existent déjà à ./expe_log/selected_data/dir_preds.csv -- abandon de l'entraînement


In [30]:
dir_pred = pd.read_csv("./expe_log/dir_preds.csv")
dir_pred = convert_to_all_numerical(dir_pred)
metrics_after_dir = getMetric(dir_pred, sensitive_attr)
compare_to_base_preds(metrics_after_dir)

0.4573 ---- base rate ---> 0.4573, (diff = 0.0000)
-0.2284 ---- SPD ---> 0.0383, (diff = 0.2667)
0.5621 ---- DI ---> 1.1150, (diff = 0.5529)
-0.1536 ---- equal_opportunity_difference ---> 0.1697, (diff = 0.3233)
-0.1680 ---- average_odds_difference ---> 0.0457, (diff = 0.2138)
-0.0639 ---- conditional_demographic_disparity ---> -0.0039, (diff = 0.0599)
0.5747 ---- smoothed_edf ---> 0.0967, (diff = -0.4781)
0.2292 ---- df_bias_amplification ---> 0.0510, (diff = -0.1782)


In [31]:
add_error_rate(dir_df, "Dir")
error_rate_df

/tmp/ipykernel_95114/3144252713.py:21: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_95114/3144252713.py:22: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,method,global,+40ans,-40ans,M,F
0,Normal,29.133333,31.083845,25.478927,25.962910,31.914894
1,Rewight,27.400000,28.834356,24.712644,26.818830,27.909887
2,Dir,30.000000,NaN,30.000000,28.815977,31.038798


In [32]:
plot_confusion_matrices_side_by_side(
    df=dir_df,
    group_column='+40ans',
    labels=["sain", "malade"],
)
# plot_confusion_matrix_by_group(dir_df["labels"], dir_df["preds"], dir_df, group_columns=["+40ans"], labels=[0, 1])

#### LFR

In [33]:
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import LFR
import pandas as pd

def apply_lfr(df, maxiter=5000, maxfun=5000):


    label_col = "labels"

    protected_attr = "+40ans"

    # ✅ Colonnes à garder pour la réparation (features + protected + label)
    dir_features = ["Patient Age", "Patient Gender", protected_attr, label_col, "WEIGHTS"] 

    # 1. Création du dataset minimal pour AIF360
    df_dir = df[dir_features].copy()

    
    dataset = BinaryLabelDataset(
        df=df_dir,
        label_names=[label_col],
        protected_attribute_names=[protected_attr]
    )

    # 2. Stockage des Patient ID pour les restaurer ensuite
    patient_ids = df["Patient ID"].astype(str).tolist()
    dataset.instance_names = [[pid] for pid in patient_ids]

    # 3. Application de LFR
    TR = LFR(
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups,
        k=5,  # Paramètre d'équilibrage des représentations
        Ax=0.001, Ay=0.1, Az=1.0,
        print_interval=500,
        verbose=1,
        seed=None
    )

    TR = TR.fit(dataset, maxiter=maxiter, maxfun=maxfun)
    repaired_dataset = TR.transform(dataset)

    # 4. Reconstruction du DataFrame réparé avec les bonnes colonnes
    repaired_df = pd.DataFrame(
        data=repaired_dataset.features,
        columns=repaired_dataset.feature_names
    )
    repaired_df[label_col] = repaired_dataset.labels

    # 5. Réinsertion des Patient ID
    repaired_df["Patient ID"] = [int(pid[0]) for pid in repaired_dataset.instance_names]

    # 6. Fusion avec imageid_df pour ajouter les chemins d’image
    imageid_df["Patient ID"] = imageid_df["Patient ID"].astype(int)
    repaired_df = repaired_df.merge(imageid_df, on="Patient ID", how="left")

    # Ajouter la colonne 'in_train' (si nécessaire)
    repaired_df["in_train"] = df["in_train"].values

    # Recalculer la colonne +40ans si nécessaire (assurer que cela soit cohérent avec l'attribut protégé)
    repaired_df["+40ans"] = (repaired_df["Patient Age"] >= 40).astype(int)

    # Sauvegarder le DataFrame final dans un fichier CSV
    repaired_df.to_csv(data_dir + "lfr_metadata.csv", index=False)

    return repaired_df


truc = apply_lfr(train_df)
truc2 = apply_lfr(df=test_df)
repaired_df = pd.concat([truc, truc2], ignore_index=True)
repaired_df.to_csv(data_dir+"lfr_metadata.csv", index=False)
# train_df contiendra le dataset transformé et fusionné avec les chemins d'images.


step: 0, loss: 1.0733071454107468, L_x: 1001.9966579662014,  L_y: 0.6937200416147816,  L_z: 0.0019384832830672105
step: 500, loss: 2.347756928971606, L_x: 58.027516184288814,  L_y: 19.63906541983342,  L_z: 0.3258228708039751
step: 1000, loss: 2.3575881385759354, L_x: 59.54439849596798,  L_y: 19.639786246692278,  L_z: 0.33406511541073924
step: 1500, loss: 0.3001046019413257, L_x: 63.66825597616366,  L_y: 0.6841293530479371,  L_z: 0.16802341066036836
step: 2000, loss: 0.2221839257692876, L_x: 121.16239101409403,  L_y: 0.6889415828784476,  L_z: 0.03212737646734881
step: 2500, loss: 0.22003032932402727, L_x: 119.38885277979387,  L_y: 0.6881651738178522,  L_z: 0.031824959162448194
step: 3000, loss: 0.21230612702717336, L_x: 125.96140653800728,  L_y: 0.6862723352172733,  L_z: 0.017717486967438776
step: 3500, loss: 0.2057876738537829, L_x: 131.7339820927544,  L_y: 0.6856006520994177,  L_z: 0.005493626551086722
step: 4000, loss: 0.20565594515094485, L_x: 131.70137769555458,  L_y: 0.68558181216

In [34]:
repaired_train_df = apply_disparate_impact_remover(train_df)
repaired_test_df =  apply_disparate_impact_remover(test_df)

repaired_df = pd.concat([repaired_train_df, repaired_test_df], ignore_index=True)

repaired_df.to_csv(data_dir+"lfr_metadata.csv", index=False)


In [35]:
train_and_predict("lfr_metadata.csv", "lfr_preds.csv")

Les prédiction existent déjà à ./expe_log/selected_data/lfr_preds.csv -- abandon de l'entraînement


In [36]:
lfr_pred = pd.read_csv(pred_output_dir+"lfr_preds.csv")
lfr_pred = convert_to_all_numerical(lfr_pred)
metrics_after_lfr = getMetric(lfr_pred, sensitive_attr)

compare_to_base_preds(metrics_after_lfr)


0.4573 ---- base rate ---> 0.4573, (diff = 0.0000)
-0.2284 ---- SPD ---> -0.0077, (diff = 0.2206)
0.5621 ---- DI ---> 0.9832, (diff = 0.4212)
-0.1536 ---- equal_opportunity_difference ---> 0.0026, (diff = 0.1563)
-0.1680 ---- average_odds_difference ---> -0.0157, (diff = 0.1523)
-0.0639 ---- conditional_demographic_disparity ---> 0.0007, (diff = 0.0646)
0.5747 ---- smoothed_edf ---> 0.0189, (diff = -0.5558)
0.2292 ---- df_bias_amplification ---> -0.0267, (diff = -0.2559)


In [37]:
add_error_rate(lfr_pred, "Lfr")
error_rate_df

/tmp/ipykernel_95114/3144252713.py:21: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/tmp/ipykernel_95114/3144252713.py:22: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,method,global,+40ans,-40ans,M,F
0,Normal,29.133333,31.083845,25.478927,25.962910,31.914894
1,Rewight,27.400000,28.834356,24.712644,26.818830,27.909887
2,Dir,30.000000,NaN,30.000000,28.815977,31.038798
3,Lfr,26.333333,NaN,26.333333,25.106990,27.409262


In [38]:
plot_confusion_matrices_side_by_side(
    df=lfr_pred,
    group_column='+40ans',
    labels=["sain", "malade"],
)
# plot_confusion_matrix_by_group(lfr_pred["labels"], lfr_pred["preds"], lfr_pred, group_columns=["+40ans"], labels=[0, 1])

## Post processing

In [39]:
df=pd.read_csv(pred_output_dir+"reweighted_preds.csv")
df.columns


Index(['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID',
       'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width',
       'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'WEIGHTS', 'preds',
       'logits_0', 'logits_1', 'labels'],
      dtype='object')

In [40]:
from aif360.algorithms.postprocessing.reject_option_classification import RejectOptionClassification

# Define privileged and unprivileged groups as dictionaries
unprivileged_groups = [{protected_attribute: unpriviliged_group}]
privileged_groups = [{protected_attribute: priviliged_group}]

metric_name = "Statistical parity difference"
metric_ub = 0.5
metric_lb = -0.5



def apply_ROC_to_preds(test_df, weights=False):
    test_df = test_df.drop(columns=["Image Index"])
    test_data = test_df[test_df['in_train']==0]

    test_dataset = BinaryLabelDataset(
        favorable_label=0,  
        unfavorable_label=1,  
        df=test_data,
        label_names=["labels"],
        protected_attribute_names=[protected_attribute]
    )


    ROC = RejectOptionClassification(
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups,
        low_class_thresh=0.0001,
        high_class_thresh=0.999,
        num_class_thresh=100,
        num_ROC_margin=50,
        metric_name=metric_name,
        metric_ub=metric_ub,
        metric_lb=metric_lb
    )

    test_dt = test_dataset.copy(deepcopy=True)
    test_dt.scores = test_data["logits_1"].values.reshape(-1, 1)
    test_dt.labels = test_data["preds"]

    ROC = ROC.fit(test_dataset, test_dt)
    df_roc_val_pred = ROC.predict(test_dt)
    post_proc_metrics=get_group_metrics(
        y_true=test_dataset.labels[:,0],
        y_pred=df_roc_val_pred .labels[:,0],
        prot_attr=test_dt.protected_attributes[:, 0],
        pos_label=1,
        sample_weight=test_dt.features[:, test_dt.feature_names.index('WEIGHTS')] if weights else None ,
    )
    return post_proc_metrics


In [41]:
print(apply_ROC_to_preds(rw_pred, weights=True))
print(apply_ROC_to_preds(dir_pred))


NameError: name 'test_df_reweighing' is not defined

In [ ]:
def cross_validate_ROC(test_df, metrics_to_try=None, class_thresholds=None, fairness_bounds=None, weights=False):
    test_df = test_df[test_df["in_train"]==0]
    test_dataset = BinaryLabelDataset(
        favorable_label=0,  
        unfavorable_label=1,  
        df=convert_to_all_numerical(test_df).select_dtypes(include=['int64', 'float64']),
        label_names=["labels"],
        protected_attribute_names=[protected_attribute]
    )

    # Default parameters
    if metrics_to_try is None:
        metrics_to_try = [
            "Statistical parity difference",
            "Equal opportunity difference",
            "Average odds difference"
        ]
    if class_thresholds is None:
        class_thresholds = [(0.01, 0.99), (0.001, 0.999)] 
    if fairness_bounds is None:
        fairness_bounds = [
            (-0.01, 0.01),
            (-0.05, 0.05),
            (-0.1, 0.1),
            (-0.25, 0.25)
        ]

  
    test_with_preds = test_dataset.copy(deepcopy=True)
    test_with_preds.labels = test_df["preds"].values.reshape(-1, 1)
    test_with_preds.scores = test_df["logits_1"].values.reshape(-1, 1)
    

    results_dfs = {}
    for metric in metrics_to_try:
        metric_results = []
        for low_thresh, high_thresh in class_thresholds:
            for lb, ub in fairness_bounds:
                try:
                   
                    # Initialize ROC with current parameters
                    ROC = RejectOptionClassification(
                        unprivileged_groups=unprivileged_groups,
                        privileged_groups=privileged_groups,
                        low_class_thresh=low_thresh,
                        high_class_thresh=high_thresh,
                        num_class_thresh=100,
                        num_ROC_margin=50,
                        metric_name=metric,
                        metric_ub=ub,
                        metric_lb=lb
                    )

                    
                    
                    # Fit ROC on both datasets (original and with predictions)
                    
                    ROC = ROC.fit(test_dataset, test_with_preds)
                    
                    print("Optimal classification threshold (with fairness constraints) = %.4f" % ROC.classification_threshold)
                    print("Optimal ROC margin = %.4f" % ROC.ROC_margin)
                    
                    # Apply the transformation to get fair predictions
                    transformed_dataset = ROC.predict(test_with_preds)
                    
                    # Calculate metrics using the transformed predictions
                    weight_column = None
                    if weights:
                        if 'WEIGHTS' in test_dataset.feature_names:
                            weight_column = test_dataset.features[:, test_dataset.feature_names.index('WEIGHTS')]
                    
                    metrics = get_group_metrics(
                        y_true=test_dataset.labels[:,0],
                        y_pred=transformed_dataset.labels[:,0],
                        prot_attr=test_dataset.protected_attributes[:, 0],
                        pos_label=1,
                        sample_weight=weight_column
                    )
                    
                    
                    result_row = {
                        'low_class_thresh': low_thresh,
                        'high_class_thresh': high_thresh,
                        'metric_lb': lb,
                        'metric_ub': ub,
                        **metrics
                    }
                    metric_results.append(result_row)
                except Exception as e:
                    print(e)
                    result_row = {
                        'low_class_thresh': low_thresh,
                        'high_class_thresh': high_thresh,
                        'metric_lb': lb,
                        'metric_ub': ub,
                        'distance': np.nan,
                        'error': str(e)
                    }
                    metric_results.append(result_row)
        
        df = pd.DataFrame(metric_results)
        df.set_index(['low_class_thresh', 'high_class_thresh', 'metric_lb', 'metric_ub'], inplace=True)
        results_dfs[metric] = df
    
    return results_dfs
results_dfs = cross_validate_ROC(dir_pred)

In [ ]:
results_dfs["Statistical parity difference"]
results_dfs["Equal opportunity difference"]
results_dfs["Average odds difference"]

In [ ]:
from aif360.algorithms.postprocessing.calibrated_eq_odds_postprocessing import CalibratedEqOddsPostprocessing



def apply_CEO(test_df):
    test_df = test_df[test_df['in_train']==0]
    cost_constraint = "fnr" # "fnr", "fpr", "weighted"
    cpp = CalibratedEqOddsPostprocessing(privileged_groups = privileged_groups,
                                        unprivileged_groups = unprivileged_groups,
                                        cost_constraint=cost_constraint,
                                        seed=42)
    
    pred_dataset = test_dataset.copy(deepcopy=True)
    pred_dataset.labels = test_df["preds"].values.reshape(-1, 1)
    pred_dataset.scores = test_df["logits_1"].values.reshape(-1, 1)
   

    cpp = cpp.fit(test_dataset, pred_dataset)
    df_ceqodds_val_pred = cpp.predict(pred_dataset)
    
    m=get_group_metrics(
        y_true=test_dataset.labels,
        y_pred=df_ceqodds_val_pred.labels[:,0],
        prot_attr=test_dataset.protected_attributes[:, 0],
        priv_group=1,
        pos_label=1,
    )
    return m

In [ ]:
print(apply_CEO(rw_pred))
print(apply_CEO(dir_pred))

## Conclusion